
# Introduction

---
Up to this point, we’ve covered the foundational concepts of machine learning. You may have already experimented with various hyperparameters to optimize model performance. In this notebook, you’ll be introduced to several advanced techniques designed to further enhance your models.


This notebook consists the following parts:

- [A: Data retrieval ](#01)
- [B: Feature engineering](#02)
- [C: Model evaluation](#03)
- [D: Ensemble learning](#04)
- [E: Pipelines](#05)
- [F: Bring it all together](#06)
- [G: Bonus: ML OPS](#07)


---

### Learning Objectives
By the end of this two weeks you will be able to:
- Understand the fundamental concepts of ensemble learning.
- Use evaluation techniques to assess models performance.
- Enhance model performance by feature engineering. 
- Build pipelines for model development and preprocessing

---

### Instructions
- Ensure you fully understand the requirements and objectives of the assignment.
- Review the notebooks refered in the tasks
- If you need additional context or clarification, please check the provided videos or background literature.
- Work through each part of the assignment methodically, ensuring all tasks are completed.
- Update your repository with your new created work

### Additional Notes:
- Do not add datafiles to your repository. Repositories with datafiles will not be accepted
- Class solutions should be delivered in python files. Not in notebooks
- When AI tools are used, you must provide proper references and explanations for how they were utilized. Failure to do so will be considered as academic fraud
- The bonus assignment are not mandatory
- Use PEP8 

Good luck!

F.Feenstra

---


<a name='01'></a>
## Part A. Data retrieval

The dataset you can use for this notebook is the lung dataset from Maastricht University. It comprises 89 non-small cell lung cancer (NSCLC) patients records who underwent surgical treatment. The study where the data is from explored the relationship between radiomic imaging features and gene expression profiles. The samples were collected through biopsies at MAASTRO Clinic in The Netherlands, and the dataset is publicly available.

The authors of the related paper discovered that a prognostic radiomic signature, which captures intra-tumor heterogeneity, is closely associated with underlying gene expression patterns. Developing a machine learning model to predict histology from the Clinical and Genetic Lung data can improve diagnostic accuracy and treatment personalization. In this notebook we will develop such a prediction model. 

**Availabel Datasets**:
- Lung metadata dataset [1]
- Gene expression dataset [2]

**Important**
<span style="background-color: lightgreen;color: black">It is also allowed to use your own dataset from your own project if this data is highly dimensional and contains genetic information.</span>

[1] [NSCLC-Radiomics-Genomics](https://wiki.cancerimagingarchive.net/display/Public/NSCLC-Radiomics-Genomics#16056856db10d39adf704eefa53e41edcf5ef41c)

[2] [Gene Expression Data - GSE58661](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE58nnn/GSE58661/matrix/)

[3] Aerts HJWL, Rios Velazquez E, Leijenaar RTH, Parmar C, Grossmann P, Carvalho S, Bussink J, Monshouwer R, Haibe-Kains B, Rietveld D, Hoebers F, Rietbergen MM, Leemans CR, Dekker A, Quackenbush J, Gillies RJ, & Lambin P. (2015). Data From NSCLC-Radiomics-Genomics. The Cancer Imaging Archive. https://doi.org/10.7937/K9/TCIA.2015.L4FRET6Z


### <span style="background-color: lightyellow;">Data retrieval task</span>
- Retrieve the data. (No cleaning needed yet)

In [ ]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, label_binarize, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,learning_curve, ParameterGrid
from sklearn.metrics import  roc_curve, auc, confusion_matrix, classification_report, accuracy_score, f1_score, roc_auc_score
from itertools import cycle
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC



In [59]:
# Read file
with gzip.open('GSE58661_series_matrix.txt.gz', 'rt') as f:
    gene_expression = pd.read_csv(f, delimiter="\t", comment='!')
    
gene_expression.to_csv('gene_expression_data.csv', index=False)
print("DataFrame created and saved successfully!")
gene_expression


In [60]:
# gene_expression.dtypes


In [61]:
# Load the lung metadata
lung_metadata = pd.read_excel('Lung3.metadata.xls')

print(lung_metadata.head())

In [62]:
# lung_metadata.dtypes

---
<a name='02'></a>
## Part B. Feature engineering

Mind you that choosing an algorithm and hyperparameter tuning might not be enough. If your data is of low quality, the algorithm will have poor performance as well. This is where feature engineering comes into play. Feature engineering involves transforming raw data into meaningful features that better represent the underlying problem to the predictive models, ultimately enhancing the model's performance. By selecting, creating, and refining features, you ensure that your data highlights the most relevant patterns and relationships, allowing the algorithm to learn more effectively. Proper feature engineering can often make the difference between a moderate model and a highly accurate one, even more so than the choice of algorithm itself. 
Possible modifications:
- creation of new features derived from original features
- selection of features
- encoding features
- log transformation 
- scaling
- dimension reduction (*e.g.* PCA)

### <span style="background-color: lightyellow;">Feature engineering Task</span>
- Review the [Study Case Feature Engineering notebook](../Study_Cases/study_case_feature_engineering.ipynb).
- Review the [Study Case RNA-seq Preparation notebook](../Study_Cases/study_case_scanpy_object.ipynb).
- Assess which data preparation and feature engineering steps could be beneficial for your dataset.
- Implement a `DataProcessor` class tailored to your dataset and test it in the cell below.
- Update your repository with a new directory named `optimization`, including the following:
    - The `DataProcessor` class as a Python file, complete with thorough documentation.
    - An evaluation document that details and justifies your choices using a well-reasoned, argumentative approach.



In [63]:
class DataProcessor:
    """
    A class for data processing and feature engineering tailored to the lung dataset.
    
    This class handles data preprocessing steps including feature selection, encoding, log transformation,
    scaling, and dimensionality reduction (PCA) as specified in the assignment criteria.
    """

    def __init__(self, gene_expression, lung_metadata):
        """
        Initializes the DataProcessor with gene expression and metadata DataFrames.
        
        Parameters:
            gene_expression (pd.DataFrame): The gene expression data.
            lung_metadata (pd.DataFrame): The metadata associated with the gene expression data.
        """
        self.gene_expression = gene_expression
        self.lung_metadata = lung_metadata.copy()
    
    def run_pca(self, variance_threshold=0.95):
        """
        Runs PCA on the gene expression data and returns the PCA DataFrame and model.

        Parameters:
            variance_threshold (float): The amount of variance to retain during PCA.

        Returns:
            pd.DataFrame: DataFrame containing PCA-transformed features.
        """
        transposed_data = self.gene_expression.iloc[1:, 1:].T.apply(pd.to_numeric, errors='coerce').dropna(axis=1)
        pca = PCA(n_components=variance_threshold)
        pca_result = pca.fit_transform(transposed_data)
        pca_df = pd.DataFrame(pca_result, columns=[f'PC{i+1}' for i in range(pca_result.shape[1])])
        return pca_df, pca
    
    def drop_single_unique_value_columns(self):
        """
        Drops columns from lung_metadata with a single unique value.
        """
        cols_to_drop = [col for col in self.lung_metadata.columns if self.lung_metadata[col].nunique() <= 1]
        self.lung_metadata.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    def encode_features(self):
        """
        Encodes categorical features in lung_metadata.
        """
        gender_mapping = {'M': 1, 'F': 0}
        stage_mapping = {'pT1': 1, 'pT2': 2, 'pT3': 3, 'pTX': 0, 'pN0': 0, 'pN1': 1, 'pNX': 0, 'pM0': 0, 'pM1': 1, 'pMX': 0}
        
        self.lung_metadata['source.location'] = self.lung_metadata['source.location'].astype('category').cat.codes
        self.lung_metadata['characteristics.tag.gender'] = self.lung_metadata['characteristics.tag.gender'].map(gender_mapping)
        
        unique_histology_values = self.lung_metadata['characteristics.tag.histology'].unique()
        histology_mapping = {value: idx for idx, value in enumerate(unique_histology_values, start=1)}
        histology_mapping['Not Available'] = 0
        self.lung_metadata['characteristics.tag.histology'] = self.lung_metadata['characteristics.tag.histology'].map(histology_mapping).fillna(0)

        self.lung_metadata['characteristics.tag.stage.primary.tumor'] = self.lung_metadata['characteristics.tag.stage.primary.tumor'].map(stage_mapping).fillna(0)
        self.lung_metadata['characteristics.tag.stage.nodes'] = self.lung_metadata['characteristics.tag.stage.nodes'].map(stage_mapping).fillna(0)
        self.lung_metadata['characteristics.tag.stage.mets'] = self.lung_metadata['characteristics.tag.stage.mets'].map(stage_mapping).fillna(0)
        
        self.lung_metadata['characteristics.tag.grade'] = self.lung_metadata['characteristics.tag.grade'].replace("Not Available", 0)
    
    def create_binary_features(self):
        """
        Creates binary features in lung_metadata.
        """
        if 'characteristics.tag.tumor.size.maximumdiameter' in self.lung_metadata.columns:
            self.lung_metadata['characteristics.tag.tumor.size.maximumdiameter'] = np.where(
                self.lung_metadata['characteristics.tag.tumor.size.maximumdiameter'] > 5, 1, 0
            )
    
    def merge_dataframes(self, pca_df):
        """
        Merges the PCA-transformed gene expression data with lung metadata.
        
        Parameters:
            pca_df (pd.DataFrame): DataFrame of PCA-transformed features.

        Returns:
            pd.DataFrame: Merged DataFrame with both PCA and lung metadata features.
        """
        self.lung_metadata.drop(columns=['title', 'CEL.file'], inplace=True, errors='ignore')
        combined_df = pd.merge(self.lung_metadata, pca_df, left_index=True, right_index=True, how='inner')
        return combined_df.dropna()
    
    def scale_and_transform(self, combined_df):
        """
        Applies log transformation and scaling to specified columns in the combined DataFrame.

        Parameters:
            combined_df (pd.DataFrame): The combined DataFrame to transform.

        Returns:
            pd.DataFrame: The transformed DataFrame.
        """
        if (combined_df['characteristics.tag.tumor.size.maximumdiameter'] > 0).all():
            combined_df['log_tumor_size'] = np.log(combined_df['characteristics.tag.tumor.size.maximumdiameter'])

        scaler = StandardScaler()
        
        # Dynamically select available PCA components for scaling
        pca_columns = [col for col in combined_df.columns if col.startswith('PC')]
        columns_to_scale = ['characteristics.tag.tumor.size.maximumdiameter'] + pca_columns
        combined_df[columns_to_scale] = scaler.fit_transform(combined_df[columns_to_scale])
        
        return combined_df



In [64]:
# Test of DataProcessor class
processor = DataProcessor(gene_expression, lung_metadata)
pca_df, _ = processor.run_pca()
processor.drop_single_unique_value_columns()
processor.encode_features()
processor.create_binary_features()
combined_df = processor.merge_dataframes(pca_df)
final_df = processor.scale_and_transform(combined_df)

print(final_df.head())

In [65]:
# Define features (X) and target variable (y)
X = final_df.drop(columns=['characteristics.tag.histology'], errors='ignore')
y = final_df['characteristics.tag.histology']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)
print("Training Labels Shape:", y_train.shape)
print("Testing Labels Shape:", y_test.shape)

### Feature Engineering Evaluation Document

objective:
to convert raw data into a more usable format It is a feature that predicts important patterns in data, making it easy for ML models to include them. Feature engineering is the process of creating, selecting, encoding, scaling, reducing, etc. features. The aim is to highlight and emphasize relevant variations…

1. Creating new features

Binary features in tumor size: We create binary features that label tumors as small or large. Depending on whether the tumor size is 5 units and up to 5 units large or large, respectively, then makes the model discriminate between different categories. with meaning more easily

Log transformation of tumor size: Using log transformation for the tumor size feature reduces the influence of a small number of extreme values. And the variance is stable...

Why: Few models can learn directly from binary features. This is because the built-in binary feature extension extends the internal feature learning. Which is why we implemented high-level changes there. Log transformation reduces the impact of severe outliers. and can stabilize the variance by making it non-linear. Relates to a linear or target distribution that is closer to the normal distribution. Record change information This makes the model more stable and interpretable. Therefore, we can directly apply the log transformation to the target distribution.

2. Selecting features

— Placing single value columns: Single value columns. (The only unique value in a column) is discarded as missing data.

Description: This model is simplified by removing non-informative features, reducing noise, and accelerating training.

3. Coding of category features

Label Coding:

We coded gender where we used 0 for women and 1 for men.

Histology and source location mapped to integers...

Tumor stages are presented sequentially. Therefore, if one stage is much more advanced than the other stage, will receive a higher value.

Why: Numerical encoding ensures that any sequence that may be nested within records will be placed in their respective locations. In order for machine learning algorithms to understand hierarchical data…

4. Sizing and standardization

StandardScale: Continuous features and PCA components are scaled, with the mean being 0 and standard being 1 for each component.

Reason: In models that use gradients. We love scaling. This helps weight the features equally and improves convergence.

5. Feature extraction — PCA for dimensionality reduction.

Principal component analysis (PCA): PCA was used to preserve 95% of the variance in gene expression data. and delete redundant information

Why: PCA makes data collection easier. Focus the model on the model with the most data. Reduce computation costs and prevent overfitting...

<a name='03'></a>
# Part C. Model Evaluation

Evaluating the performance of a machine learning model goes beyond just looking at accuracy, as accuracy alone can be misleading, especially in cases where the dataset is imbalanced or where different types of errors have different consequences
- **Detecting Overfitting/Underfitting**: A learning curve can help you understand whether your model is overfitting (performing well on training data but poorly on validation data) or underfitting (performing poorly on both training and validation data)
- **ROC**: The ROC curve shows how well your model distinguishes between classes. It helps in selecting the optimal threshold for classification decisions, particularly when the cost of false positives and false negatives differs significantly. ROC curves are often used to compare models. 
- **Confusion matrix**: From the confusion matrix, you can derive other important metrics like precision, recall, F1-score, and specificity, which give a better understanding of how your model is performing across different classes


### <span style="background-color: lightyellow;">Evaluation Task</span>
- Review the documentation for `sklearn.metrics` and `learning_curve`.
- Select appropriate metrics for your dataset, including accuracy and indicators of overfitting or underfitting.
- Implement functions to compute and assess these metrics. It is allowed to use libraries.
- Add the evaluation functions as a Python module to your repository.
- Update the evaluation documentation to clearly explain your choices for the evaluation metrics and the rationale behind them.

See also: [Model evaluation video](https://video.hanze.nl/media/model-evaluation/0_gybpnhq7)


In [ ]:
class ModelEvaluator:
    def __init__(self, estimator, X_train, y_train, X_test, y_test):
        """
        Initialize the ModelEvaluator with a model and training/test data.
        
        Parameters:
            estimator: Trained model estimator
            X_train (pd.DataFrame or np.array): Training feature data
            y_train (pd.Series or np.array): Training target data
            X_test (pd.DataFrame or np.array): Testing feature data
            y_test (pd.Series or np.array): Testing target data
        """
        self.estimator = estimator
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test

    def plot_learning_curve(self, train_sizes=np.linspace(0.1, 1.0, 5)):
        """
        Plot the learning curve for the estimator.
        
        Parameters:
            train_sizes (np.array): Proportion of training examples used to generate learning curves.
        """
        train_sizes, train_scores, test_scores = learning_curve(
            self.estimator, self.X_train, self.y_train, train_sizes=train_sizes, cv=5
        )
        train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
        test_mean, test_std = test_scores.mean(axis=1), test_scores.std(axis=1)

        plt.figure()
        plt.plot(train_sizes, train_mean, 'o-', label="Training score")
        plt.plot(train_sizes, test_mean, 'o-', label="Cross-validation score")
        plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1)
        plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1)
        plt.xlabel("Training examples")
        plt.ylabel("Score")
        plt.legend(loc="best")
        plt.title("Learning Curve")
        plt.show()

    def plot_multiclass_roc_curve(self):
        """
        Plot ROC curve for multiclass classification.
        """
        y_test_bin = label_binarize(self.y_test, classes=np.unique(self.y_test))
        n_classes = y_test_bin.shape[1]
        y_score = self.estimator.predict_proba(self.X_test)

        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        plt.figure()
        colors = cycle(['aqua', 'darkorange', 'cornflowerblue'])
        for i, color in zip(range(n_classes), colors):
            plt.plot(fpr[i], tpr[i], color=color, lw=2,
                     label=f'Class {i} (area = {roc_auc[i]:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve for Multi-Class Classification")
        plt.legend(loc="best")
        plt.show()

    def plot_confusion_matrix(self):
        """
        Plot the confusion matrix for the model's predictions.
        """
        y_pred = self.estimator.predict(self.X_test)
        cm = confusion_matrix(self.y_test, y_pred)
        plt.figure()
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title("Confusion Matrix")
        plt.colorbar()
        tick_marks = np.arange(len(cm))
        plt.xticks(tick_marks, rotation=45)
        plt.yticks(tick_marks)

        fmt = 'd'
        thresh = cm.max() / 2
        for i, j in np.ndindex(cm.shape):
            plt.text(j, i, format(cm[i, j], fmt), horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()
        
        print("Classification Report:")
        print(classification_report(self.y_test, y_pred))




In [67]:
# Initialize a simple classifier (Logistic Regression)
estimator = LogisticRegression(max_iter=1000, random_state=42)
estimator.fit(X_train, y_train)

# Testing evaluation class
evaluator = ModelEvaluator(estimator, X_train, y_train, X_test, y_test)
evaluator.plot_learning_curve()
evaluator.plot_multiclass_roc_curve()
evaluator.plot_confusion_matrix()

Model performance evaluation documentation
1. Over-device/under-device detection: Learning curve
objective:
The purpose of the learning curve is to help identify if the model is overfitting. (high training accuracy but low verification) or less appropriate (It has low accuracy in both training and validation.) Overfitting indicates that the model captures more noise than the underlying data format. While a lower installation indicates that the relevant format is not captured...

reason:
Learning curves are plotted using training scores and cross-validation at different training sizes. This visualization helps with the following:

Overfitting detection: If the training score is high and the cross-validation score is low It shows that the model performs well on the training data. but cannot be generalized. It recommends that there be too many installations...
Underfitting detection: If the training and validation scores are low indicates that the model is not appropriate. This may be due to the complexity of the model or lack of sufficient training data...
In this case, we observe that the training scores remain high. while the cross-validation score decreased. This indicates the possibility of over-installation. This indicates that the model may need to be normalized or refined to improve generality.

2. ROC curves for multi-class classification.
objective:
The ROC graph evaluates how well the model discriminates between classes. In binary classification, the ROC curve plots the true positive rate (sensitivity) against the false positive rate (1 - specificity), but in a multiclass setting we use the One-vs-Rest (OvR) method.
reason:
For multi-class problems, separate ROC curves are plotted for each class. This shows that each class can be differentiated from other classes. How good is it all? This allows us to:

Compare class differences: The area under each ROC curve (AUC) indicates how well the model distinguishes each class from the rest.
Evaluate the ideal threshold: ROC curves are useful when selecting the optimal threshold for classification decisions. Especially if there is a significant change in the value of false positives and negatives...
The ROC curves in this evaluation show different AUC scores across classes. This indicates that some classes are easier to distinguish than others. Improving class separation with lower AUC scores may require more targeted feature engineering or other models.

3. Report confusion and classification
objective:
Confusion matrices provide detailed insights into how well the model performs across categories. It shows true positives, false positives, true negatives, and false negatives for each category...

reason:
Confusion matrices are especially useful for:

Class Level Assessment: Confusion matrices differ from accuracy in that they provide a single measurement. This allows us to examine each class separately. This is important in unbalanced data sets...
The main measures obtained: from the confusion matrix. We calculated precision, recall, F1 score, and specificity. It provides insights into the model's performance across categories...
The classification report also specifies these criteria:

Accuracy: Specifies the number of correct positive predictions of the model. If false positives are significant This indicator is also important.
Recall: Shows the number of true positives that were correctly identified by the model. This is important in situations where false negatives are large.
F1 Score: Connected

<a name='04'></a>
## Part D. Ensemble learning
Ensemble learning is a powerful machine learning technique that combines the predictions of multiple models to improve overall performance, robustness, and accuracy. Rather than relying on a single model, ensemble learning methods aggregate the results of several models—often called "weak learners"—to produce a stronger predictive model. The key idea behind ensemble learning is that by combining models, the weaknesses of individual models can be offset, leading to better generalization on unseen data. Popular ensemble ML algorithms are the `Random Forest` and `XGBoost`. 
Here’s an improved version of the introduction:

### voting algorithms
Voting algorithms in ensemble learning combine the predictions of multiple classifiers to make a final decision, typically based on the consensus or weighted agreement among models. Two primary types of voting are commonly used: hard voting, where the final prediction is determined by the majority vote, and soft voting, which uses the weighted average of predicted probabilities to determine the outcome. Several sites explain the hard and soft voting algorithm. A clear explanation can be found on https://www.baeldung.com/cs/hard-vs-soft-voting-classifiers


### <span style="background-color: lightyellow;">Ensemble Task</span>
- Review the [Study Case notebook on ensemble learning](..Study_Cases/study_case_bagging_boosting.ipynb).
- Try three different algorithms for classification of your data label.
- Implement a hard and soft voting algorithm for model aggregation.
- Compare the performance of the voting algorithm with that of a boosting or bagging algorithm.
- Update your repository with the voting algorithm class

See also: [ensemle learning video](https://video.hanze.nl/media/Ensemble/0_sue5v33g)


In [69]:
class EnsembleModel:
    def __init__(self):
        """
        Initializes an EnsembleModel with a set of base models.
        """
        self.models = {
            "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
            "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
            "Bagging": BaggingClassifier(estimator=LogisticRegression(), n_estimators=100, random_state=42),
            "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
            "LogisticRegression": LogisticRegression(),
            "GaussianNB": GaussianNB(),
            "SVM": SVC(probability=True, random_state=42)
        }
        self.voting_clf_hard = None
        self.voting_clf_soft = None

    def fit_base_models(self, X_train, y_train):
        """
        Fits each of the base models on the training data.
        """
        for name, model in self.models.items():
            model.fit(X_train, y_train)
            print(f"{name} trained.")

    def vote_hard(self, X_train, y_train, X_test, y_test):
        """
        Performs hard voting using the base models and evaluates the result.
        """
        self.voting_clf_hard = VotingClassifier(
            estimators=[(name, model) for name, model in self.models.items()],
            voting='hard'
        )
        self.voting_clf_hard.fit(X_train, y_train)
        y_pred = self.voting_clf_hard.predict(X_test)
        print("\n--- Hard Voting ---")
        self.evaluate_metrics(y_test, y_pred)

    def vote_soft(self, X_train, y_train, X_test, y_test):
        """
        Performs soft voting using the base models and evaluates the result.
        """
        self.voting_clf_soft = VotingClassifier(
            estimators=[(name, model) for name, model in self.models.items()],
            voting='soft'
        )
        self.voting_clf_soft.fit(X_train, y_train)
        y_pred = self.voting_clf_soft.predict(X_test)
        print("\n--- Soft Voting ---")
        self.evaluate_metrics(y_test, y_pred, model=self.voting_clf_soft, X_test=X_test)

    def evaluate_metrics(self, y_test, y_pred, model=None, X_test=None):
        """
        Evaluates model performance and prints metrics including accuracy, F1 score, ROC AUC, and classification report.
        """
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
        print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted'):.2f}")

        if model is not None and X_test is not None:
            try:
                print(f"ROC AUC Score: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.2f}")
            except ValueError:
                print("ROC AUC Score calculation failed. Ensure binary or multiclass format is correct.")

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))




In [70]:
# Initialize and train the ensemble model
ensemble_model = EnsembleModel()
ensemble_model.fit_base_models(X_train, y_train)

# Perform hard and soft voting
ensemble_model.vote_hard(X_train, y_train, X_test, y_test)
ensemble_model.vote_soft(X_train, y_train, X_test, y_test)

In [ ]:
# # Compare with bagging and boosting
# def compare_with_bagging_boosting(X_train, y_train, X_test, y_test):
#     # Bagging using RandomForest
#     bagging_model = RandomForestClassifier(n_estimators=100, random_state=42)
#     bagging_model.fit(X_train, y_train)
#     y_pred_bagging = bagging_model.predict(X_test)
#     print("\n--- Bagging (Random Forest) ---")
#     evaluate_metrics(y_test, y_pred_bagging, model=bagging_model, X_test=X_test)

#     # Boosting using GradientBoosting
#     boosting_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
#     boosting_model.fit(X_train, y_train)
#     y_pred_boosting = boosting_model.predict(X_test)
#     print("\n--- Boosting (Gradient Boosting) ---")
#     evaluate_metrics(y_test, y_pred_boosting, model=boosting_model, X_test=X_test)

# # Compare with bagging and boosting
# compare_with_bagging_boosting(X_train, y_train, X_test, y_test)

### Summary of the overall learning assessment.

#### objective
Improving prediction accuracy for multiclass classification problems using cluster learning methods. We use hard and soft voting without using any pre-built functions. and compare these with Bagging (RandomForest) and Boosting (GradientBoosting) approaches.

#### Data set
The data set is unbalanced with very few examples in some classes This affects classification criteria and interpretation.

####Models and techniques
1. **Basic Model**: Logistic Regression Naive Bayes K-nearest neighbor decision tree. and support vector machines (for soft voting)
2. **Voting Guidelines**:
   - **Heavy voting**: Most votes on each model's predictions.
   - **Light Voting**: Collect probability scores from each model. and select the class with the highest average probability.
3. **Backing (random forest)**: Combine predictions from multiple decision trees to improve robustness.
4. **Gradient boosting**: Train the model gradually, paying attention to errors in previous models. to improve accuracy

#### Evaluation indicators
1. **Accuracy**: The percentage of the sample correctly classified.
2. **F1 Score**: Balances precision and recall. This is useful for unbalanced data.
3. **ROC AUC Score**: Measures the model's ability to discriminate between classes.
4. **Classification Report**: Provides precision, recall, and F1 score per class.
5. **Confusion Matrix**: Shows actual vs. predicted classification. It shows the incorrect classification per class.

#### result

1. **Hard Vote**:
   - **Accuracy**: 50%
   - **F1 score**: 0.39
   - **Insights**: Moderate performance with low precision/recall for minority categories.

2. **Informal voting**:
   - **Accuracy**: 56%
   - **F1 score**: 0.38
   - **Insights**: Minor improvements over the denominator.

3. **Wrapping (random forest)**: .
   - **Accuracy**: 50%
   - **F1 score**: 0.38
   - **Insights**: Similar to the voting method. But it handles class distribution a little better.

4. **Acceleration (gradient acceleration)**:
   - **Accuracy**: 44%
   - **F1 score**: 0.40
   - **Insights**: Less accuracy but slightly higher F1 score. This shows some improvement for the minority class.

#### advice
1. **Class Balancing**: Techniques like SMOTE can improve results for minority classes.
2. **Custom Voting Weights**: Adjust voting weights slightly to prioritize model strength.
3. **Alternative methods**: Using XGBoost or other techniques such as stacking may provide better results.

#### Gathering
Soft voting achieved the highest accuracy. But overall performance points to the need for additional tuning and balancing to effectively address class imbalances...


<a name='05'></a>
## Part E. Pipelines
The sklearn pipeline function is a tool in machine learning that simplifies the workflow by encapsulating all the steps involved in a single object. It offers advantages such as simplicity, reproducibility, efficiency, flexibility, and integration. The Pipeline is built using a list of (key, value) pairs, where the key is a string containing the name you want to give this step and value is an estimator object (the method to be executed).

### <span style="background-color: lightyellow;">Pipeline Task</span>
- Read the [Study Case notebook for a pipeline functions](..Study_Cases/study_case_pipeline.ipynb) to understand the principle of the pipeline function
- Implement a pipeline which prepares and classifies data 
- Use a `GridSearchCV` object with the `Pipeline` object and a parameter grid to optimize choose the best hyper parameters

In [ ]:
class CustomPipeline:
    def __init__(self, steps):
        self.steps = steps
        self.models = {}
    
    def fit(self, X, y):
        for name, estimator in self.steps:
            if name == "encoder":
                X = estimator.fit_transform(X)
            elif name == "scaler" or name == "pca":
                X = estimator.fit_transform(X)
            else:
                estimator.fit(X, y)
                self.models[name] = estimator
    
    def predict(self, X):
        for name, estimator in self.steps:
            if name == "encoder":
                X = estimator.transform(X)
            elif name == "scaler" or name == "pca":
                X = estimator.transform(X)
        return self.models["classifier"].predict(X)

    def set_params(self, params):
        for param, value in params.items():
            step_name, param_name = param.split("__")
            for i, (name, estimator) in enumerate(self.steps):
                if name == step_name:
                    setattr(estimator, param_name, value)
                    self.steps[i] = (name, estimator)

# Define grid search
def grid_search_cv(X_train, y_train, pipeline, param_grid):
    best_score = 0
    best_params = None
    for params in ParameterGrid(param_grid):
        pipeline.set_params(params)
        pipeline.fit(X_train, y_train)
        score = accuracy_score(y_train, pipeline.predict(X_train))
        if score > best_score:
            best_score = score
            best_params = params
    return best_score, best_params


In [71]:

# Define custom pipeline and update parameter grid
pipeline_steps = [
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=3)),  # Adjust PCA components
    ("classifier", RandomForestClassifier(class_weight="balanced", random_state=42))
]

pipeline = CustomPipeline(steps=pipeline_steps)

param_grid = {
    "pca__n_components": [2, 3],  # Lower PCA components
    "classifier__n_estimators": [30, 50],  # Fewer estimators
    "classifier__max_depth": [5, 10],  # Limit tree depth
    "classifier__min_samples_split": [10, 20]  # Larger minimum split
}

# Perform updated grid search
best_score, best_params = grid_search_cv(X_train, y_train, pipeline, param_grid)

print("Best score from grid search:", best_score)
print("Best parameters:", best_params)

# Evaluate on test data
y_pred = pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_accuracy)
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

### Pipeline Evaluation Summary

#### objective
To create an improved pipeline for efficient data pre-processing, reduction, and classification. The purpose of this pipeline is to streamline the workflow and improve model performance. By adjusting parameters through table search

#### Pipe components
1. **One-hot encoding**: Convert category features to binary columns. which classifier can be used
2. **Normal scaling**: Numerical features that are normalized to increase model performance.
3. **PCA**: Reduced size while maintaining highly informative features.
4. **RandomForestClassifier**: Used for robustness and flexibility in handling complex data.

#### Hyperparameter tuning
Network search is used to optimize parameters such as PCA components, number of trees, tree depth. and the minimum sample required for node extraction. The best criteria are identified:
- `pca__n_component`:3
- `Classifier__n_estimator`:50
- `Maximum__depth__of_classification`:
- `Classifier__minute_sampling_split`:10

#### result
- **Training accuracy**: 71.83%
- **Test accuracy**: 0%
- **Insights**: Even with customizations But the model had too much training data. and encountered problems with test predictions. This may be due to class imbalance.

#### challenge
1. **Class Imbalance**: The model experiences problems with a minority of classes. This results in poor test performance.
2. **Overfitting**: High training accuracy but zero testing accuracy indicates that the model captures more noise than the normal model.

#### Conclusion
The pipeline provided a structured approach but revealed challenges with class imbalance and overfitting. Addressing these issues and exploring ensemble methods may lead to better test performance and generalizability.

## <span style="background-color: lightyellow;">Bring it all together</span>

By now, you've developed code snippets for model evaluation, optimization, and data improvement. Now, leverage these skills to build a classification model using the Clinical and Genetic Lung data. Make sure that you log your experiments. 

Once you're satisfied with the model, upload the relevant code to your repository and or refactor code with new insights. Furthermore, take a moment to reflect on its applicability and potential real-world impact. Update your evaluation document(s) with these findings in your repository. 

## Bonus: ML for operations

If we intend to deploy the model in a real-world application, it's more efficient to save and reuse the trained model rather than retraining it each time. 

- Read the blog: https://neptune.ai/blog/saving-trained-model-in-python
- Write three python files 
    1) a train_model python file 
    2) a use_model python file
    3) a retrain_model python file that adds new data to the original training data and updates the model
- Update your repository
